# Multi-Target DTI Prediction: Moringa oleifera Phytochemicals vs MAO-B, AChE, BACE1

In [ ]:
!git clone https://github.com/kexinhuang12345/DeepPurpose.git
%cd /content/DeepPurpose
!pip install -e . -q --no-deps
!pip install rdkit pubchempy -q
!pip install dgllife lifelines pandas-flavor wget subword-nmt -q
!pip install git+https://github.com/bp-kelley/descriptastorus -q

In [ ]:
from DeepPurpose import utils, DTI
print("DeepPurpose imported successfully")

In [ ]:
import requests

targets = {
    "MAOB": "P27338",
    "AChE": "P22303",
    "BACE1": "P56817"
}

sequences = {}
for name, acc in targets.items():
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
    r = requests.get(url)
    fasta = r.text
    seq = "".join(fasta.split("\n")[1:]).strip()
    sequences[name] = seq
    print(f">{name}|{acc}  length={len(seq)}")

In [ ]:
import pubchempy as pcp

compound_names = [
    "3-p-Coumaroylquinic acid",
    "Isolariciresinol",
    "4-Caffeoylquinic acid",
    "Rutin",
    "Luteolin",
    "Kaempferol",
    "Quercetin",
    "Secoisolariciresinol",
    "Apigenin",
    "Myricetin",
    "Medioresinol",
    "O-Coumaric Acid",
    "Selegiline"
]

drugs = {}
failed = []
for name in compound_names:
    try:
        result = pcp.get_compounds(name, 'name')
        if result:
            drugs[name] = result[0].canonical_smiles
            print(f"{name}: {result[0].canonical_smiles}  (CID {result[0].cid})")
        else:
            failed.append(name)
    except Exception as e:
        failed.append(name)
        print(f"FAILED: {name} -> {e}")

print(f"\nResolved {len(drugs)} of {len(compound_names)} compounds.")
if failed:
    print("Need manual SMILES for:", failed)

In [ ]:
drug_list, target_list, drug_names, target_names = [], [], [], []

for dname, smiles in drugs.items():
    for tname, tseq in sequences.items():
        drug_list.append(smiles)
        target_list.append(tseq)
        drug_names.append(dname)
        target_names.append(tname)

print(f"Total pairs to predict: {len(drug_list)}")

In [ ]:
X_pred = utils.data_process(
    X_drug=drug_list,
    X_target=target_list,
    y=[0]*len(drug_list),
    drug_encoding='Morgan',
    target_encoding='CNN',
    split_method='no_split'
)

model = DTI.model_pretrained(model='Morgan_CNN_BindingDB')
y_pred = model.predict(X_pred)

print("Number of predictions:", len(y_pred))

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Compound": drug_names,
    "Target": target_names,
    "Predicted_Score": y_pred
})

pivot = results_df.pivot(index="Compound", columns="Target", values="Predicted_Score")
print(pivot)

results_df.to_csv("DTI_predictions_long.csv", index=False)
pivot.to_csv("DTI_predictions_by_target.csv")

In [ ]:
from google.colab import files
files.download("DTI_predictions_long.csv")
files.download("DTI_predictions_by_target.csv")